In [15]:
# Installing
%pip install -q -U transformers datasets peft trl bitsandbytes accelerate

Note: you may need to restart the kernel to use updated packages.


In [16]:
print('🚀 Upgrading torchao to a compatible version...')
%pip install --upgrade torchao
print('✅ torchao upgraded successfully!')

🚀 Upgrading torchao to a compatible version...
Note: you may need to restart the kernel to use updated packages.
✅ torchao upgraded successfully!


In [17]:
# Reinstall pyarrow, pandas, and datasets to resolve potential binary incompatibility issues.
# This is a common fix for 'pyarrow.lib.IpcReadOptions size changed' error.
# After running this cell, please restart your Colab runtime.
%pip install --upgrade --no-cache-dir pyarrow pandas datasets

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

Note: you may need to restart the kernel to use updated packages.


In [18]:

print("📡 Loading math dataset...")
dataset = load_dataset("openai/gsm8k", "main", split="train[:4000]") # Using a subset for fast
print(dataset)

📡 Loading math dataset...
Dataset({
    features: ['question', 'answer'],
    num_rows: 4000
})


In [19]:
# # 2. Setup 4-bit quantization
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,  # Double-check that this is exactly torch.float16
#     bnb_4bit_use_double_quant=True,
# )

In [ ]:
# 3. Load Base Foundation Model (Using Qwen2.5-1.5B-Instruct as our backbone)
model_id = "Qwen/Qwen2.5-1.5B-Instruct" # A free, open weight backbone
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    dtype=torch.float16
)
print("done loading model")
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
dtypes = set(p.dtype for p in model.parameters())
print(dtypes)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

KeyboardInterrupt: 

In [ ]:
# 4. Map the dataset columns to a unified training format
def format_prompts(batch):
    formatted_texts = []
    for q, a in zip(batch["question"], batch["answer"]):
        # Formatting strictly as Instruction -> Response
        text = f"### Instruction:\n{q}\n\n### Response:\n{a}</s>"
        formatted_texts.append(text)
    return {"text": formatted_texts}

dataset = dataset.map(format_prompts, batched=True)
# ---> FIX HERE: Extract the dataset from the DatasetDict if necessary <---
if isinstance(dataset, dict) and "train" in dataset:
    dataset = dataset["train"]
dataset = dataset.train_test_split(test_size=0.1)
print(dataset)

In [ ]:
# 5. Define LoRA Configuration (Targets specific mathematical weight adapters)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
training_args = SFTConfig(
    output_dir="./openai-gsm8k",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    # evaluation_strategy="epoch",
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    save_strategy="epoch",
    dataset_text_field="text",
    max_length=512,

    # ---> CRITICAL T4 GPU ALIGNMENT <---
   fp16=False,
    bf16=False,

    dataloader_drop_last=True,
    optim="adamw_torch",
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer,
)

In [ ]:
print("🚀 Fine-tuning your math model...")
trainer.train()

# 8. Save the pixel-perfect adapters locally
trainer.model.save_pretrained("./openai-gsm8k")
tokenizer.save_pretrained("./openai-gsm8k")
print("🎉 Math model adapters saved successfully!")

### Saving and Loading Model from Google Drive

To make your fine-tuned model persist across runtime sessions, we'll save it to Google Drive. This involves mounting your Drive, specifying a save path, and then copying the saved model and tokenizer files.


In [ ]:
# # 1. Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)


In [ ]:
# 2. Define the path in Google Drive
drive_model_path = "./models/gsm8k"

# Ensure the directory exists
import os
os.makedirs(drive_model_path, exist_ok=True)

# 3. Save the model and tokenizer to Google Drive
trainer.model.save_pretrained(drive_model_path)
tokenizer.save_pretrained(drive_model_path)

print(f"🎉 Math model adapters and tokenizer saved to Google Drive at: {drive_model_path}")

Now, even if your Colab runtime restarts, you can load your model directly from Google Drive. Here's how to load it and use it for inference:

In [ ]:
# Define local and Google Drive model paths
local_model_path = "./models/gsm8k"
drive_model_path = "/content/drive/MyDrive/openai-gsm8k"

# Determine which path to use for loading
import os
if os.path.exists(local_model_path):
    model_load_path = local_model_path
    print(f"Loading model from local path: {model_load_path}")
# else:
#     # Mount Google Drive if not already mounted
#     from google.colab import drive
#     if not os.path.exists('/content/drive'):
#         drive.mount('/content/drive', force_remount=True)
#     model_load_path = drive_model_path
#     print(f"Loading model from Google Drive path: {model_load_path}")

In [ ]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 2. Setup 4-bit quantization (Essential for Free Tier GPUs)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # Double-check that this is exactly torch.float16
    bnb_4bit_use_double_quant=True,
)

loaded_model = AutoModelForCausalLM.from_pretrained(
    model_load_path, # Use the dynamically determined path
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)
loaded_tokenizer = AutoTokenizer.from_pretrained(model_load_path)

math_solver_from_drive = pipeline(
    "text-generation",
    model=loaded_model,
    tokenizer=loaded_tokenizer
)

# 5. Test the model loaded from Google Drive
test_prompt = "### Instruction:\nFind the value of x if 3x + 7 = 22.\n\n### Response:\n"

results_from_drive = math_solver_from_drive(test_prompt, max_new_tokens=50, do_sample=False)
print(results_from_drive[0]["generated_text"])

In [ ]:
math_solver_from_drive = pipeline(
    "text-generation",
    model=loaded_model,
    tokenizer=loaded_tokenizer
)

# 5. Test the model loaded from Google Drive
test_prompt = "### Instruction:\nMark has a box of 40 chocolates. He eats 5 chocolates, gives half of the remaining chocolates to his sister, and then buys 10 more chocolates the next day. How many chocolates does Mark have now?\n\n### Response:\n"

results_from_drive = math_solver_from_drive(test_prompt, max_new_tokens=500, do_sample=False)
print(results_from_drive[0]["generated_text"])